# Chronicle notebook 02: session workflow\n\nThis notebook walks through a minimal project-backed Chronicle workflow.

In [ ]:
from pathlib import Path\nfrom pprint import pprint\nimport tempfile\n\nfrom chronicle.store.project import create_project\nfrom chronicle.store.session import ChronicleSession

In [ ]:
project_path = Path(tempfile.mkdtemp(prefix="chronicle_nb_"))\ncreate_project(project_path)\nproject_path

In [ ]:
with ChronicleSession(project_path) as session:\n    _, investigation_uid = session.create_investigation(\n        "Notebook walkthrough",\n        description="Notebook-created investigation for scorer exploration",\n        actor_id="notebook",\n        actor_type="tool",\n    )\n\n    _, evidence_uid = session.ingest_evidence(\n        investigation_uid,\n        b"Quarterly filing says revenue reached $1.2M in Q1 2024.",\n        "text/plain",\n        original_filename="q1_report.txt",\n        actor_id="notebook",\n        actor_type="tool",\n    )\n\n    _, span_uid = session.anchor_span(\n        investigation_uid,\n        evidence_uid,\n        "text_offset",\n        {"start_char": 0, "end_char": 38},\n        quote="Quarterly filing says revenue reached",\n        actor_id="notebook",\n        actor_type="tool",\n    )\n\n    _, claim_uid = session.propose_claim(\n        investigation_uid,\n        "Acme Corp reported $1.2M revenue in Q1 2024.",\n        actor_id="notebook",\n        actor_type="tool",\n    )\n\n    session.link_support(\n        investigation_uid,\n        span_uid,\n        claim_uid,\n        rationale="Span contains the exact revenue statement.",\n        actor_id="notebook",\n        actor_type="tool",\n    )\n\n    score = session.get_defensibility_score(claim_uid)\n\nprint("investigation_uid:", investigation_uid)\nprint("claim_uid:", claim_uid)\npprint(score)